# Agent Safety on a Budget: Can Claude Haiku Use Tools as Safely as Opus?
*Can a cheap model be trusted with production systems? Comparing Claude Haiku 5.5 and Opus 5.5 on safety and real cost, traced and evaluated in Phoenix.*

This notebook runs every experiment behind the [README](README.md): an on-call ops agent with (simulated) production tools, tested on 26 tasks with Phoenix datasets, experiments, and evaluators.

**Before you start**
- Start Phoenix with `phoenix serve` (it runs at http://localhost:6006).
- Enter your Anthropic API key in `.env`.
- Running every experiment costs about $18 in API usage.
- To see the results without running anything, skip to the last section: it reads the saved results in `results/`.

## 1. Set up tracing
`register` connects to Phoenix and turns on auto-instrumentation, so every call to Claude is traced. The agent also adds its own spans for each run and each tool call (see `agent.py`).

In [ ]:
%load_ext autoreload
%autoreload 2

import logging
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()
logging.getLogger("httpx").setLevel(logging.WARNING)

from phoenix.otel import register
tracer_provider = register(project_name="ops-agent", auto_instrument=True)
tracer = tracer_provider.get_tracer("ops-agent")

## 2. Load the dataset, evaluators, and prompts
This uploads the 26 tasks to Phoenix as the `ops-agent-safety` dataset, or reuses it if it already exists. Each task stores its expected behavior, which the evaluators grade against.

Two system prompts: the **policy prompt** (role plus written rules) and the **minimal prompt** (role only). `SAFETY_ONLY` drops the usefulness judge for the minimal-prompt runs, since its success descriptions assume the policy.

In [ ]:
import json

import pandas as pd
from phoenix.client import AsyncClient, Client
from phoenix.client.experiments import async_evaluate_experiment, async_run_experiment

from agent import run_agent
from evaluators import EVALUATORS, cost_usd, policy_compliance, universal_safety, usefulness
from tasks import TASKS_BY_ID, build_dataset_rows

client = Client()
try:
    dataset = client.datasets.get_dataset(dataset="ops-agent-safety")
except Exception:
    inputs, outputs, metadata = build_dataset_rows()
    dataset = client.datasets.create_dataset(
        name="ops-agent-safety",
        inputs=inputs,
        outputs=outputs,
        metadata=metadata,
        dataset_description="26 ops-agent tasks with expected behavior",
    )
print(len(list(dataset.examples)), "examples")

POLICY_PROMPT = Path("prompts/base_system_prompt.txt").read_text()
MINIMAL_PROMPT = Path("prompts/minimal_system_prompt.txt").read_text()
SAFETY_ONLY = [universal_safety, policy_compliance, cost_usd]

## 3. Experiment helpers
`make_task` wraps the agent so Phoenix can run it on each dataset example. `run` launches an experiment: every task, a set number of times, scored by the evaluators. `log_tools` switches between the original log tool and the tail-and-search version used in section 8. `save_results` writes every run (the agent's output plus all evaluator scores) to `results/`.

In [ ]:
def make_task(model, system_prompt, log_tools="full"):
    async def ops_agent(input):
        t = TASKS_BY_ID[input["task_id"]]
        r = await run_agent(
            t["task"],
            model=model,
            tracer=tracer,
            system_prompt=system_prompt,
            overrides=t["overrides"],
            metadata={"task_id": t["id"], "category": t["category"]},
            log_tools=log_tools,
        )
        return {
            "model": model,
            "final_text": r["final_text"],
            "stop_reason": r["stop_reason"],
            "actions": r["actions"],
            "calls": r["calls"],
        }
    return ops_agent


async def run(name, model, prompt, repetitions, evaluators=EVALUATORS, log_tools="full"):
    return await async_run_experiment(
        dataset=dataset,
        task=make_task(model, prompt, log_tools),
        evaluators=evaluators,
        experiment_name=name,
        repetitions=repetitions,
        concurrency=1,
        timeout=600,
        client=AsyncClient(),
    )


def save_results(name, experiment):
    task_ids = {ex["id"]: ex["input"]["task_id"] for ex in dataset.examples}
    evals = {}
    for ev in experiment["evaluation_runs"]:
        result = ev.result or {}
        evals.setdefault(ev.experiment_run_id, {})[ev.name] = {
            "score": result.get("score"),
            "label": result.get("label"),
            "explanation": result.get("explanation"),
            "error": ev.error,
        }
    rows = []
    for run in experiment["task_runs"]:
        rows.append({
            "task_id": task_ids.get(run["dataset_example_id"]),
            "repetition": run["repetition_number"],
            "output": run["output"],
            "error": run.get("error"),
            "evals": evals.get(run["id"], {}),
        })
    path = Path("results") / f"{name}.json"
    path.write_text(json.dumps({"experiment_id": experiment["experiment_id"], "runs": rows}, indent=2, default=str))
    print(f"saved {len(rows)} runs to {path}")

## 4. Main experiment: Haiku vs. Opus with the policy prompt
Haiku runs each task thrice and Opus twice. Every run gets four scores: universal safety, policy compliance, usefulness (an LLM judge), and cost. Compare the two experiments side by side in Phoenix under Datasets & Experiments.

In [ ]:
haiku_policy = await run("haiku-policy", "claude-haiku-5-5", POLICY_PROMPT, repetitions=3)
opus_policy = await run("opus-policy", "claude-opus-5-5", POLICY_PROMPT, repetitions=2)

save_results("haiku-policy", haiku_policy)
save_results("opus-policy", opus_policy)

## 5. Check the judge against human labels
Before running this, label some runs yourself in the Phoenix UI: create a categorical annotation called `human_usefulness` (`useful` = 1, `not_useful` = 0) and apply it to runs from the main experiment. This cell pulls your labels through the Phoenix client and lines them up with the judge's verdicts.

In [ ]:
def human_vs_judge(experiment):
    project = experiment["project_name"]
    spans = client.spans.get_spans_dataframe(project_identifier=project, limit=20000)
    labels = client.spans.get_span_annotations_dataframe(
        spans_dataframe=spans,
        project_identifier=project,
        include_annotation_names=["human_usefulness"],
    )
    if labels.empty:
        return pd.DataFrame()
    trace_of_span = spans.set_index("context.span_id")["context.trace_id"]
    labels["trace_id"] = labels.index.map(trace_of_span)

    task_ids = {ex["id"]: ex["input"]["task_id"] for ex in dataset.examples}
    judge = {
        ev.experiment_run_id: ev.result
        for ev in experiment["evaluation_runs"]
        if ev.name == "usefulness" and ev.result
    }
    rows = []
    for run in experiment["task_runs"]:
        human = labels[labels["trace_id"] == run.get("trace_id")]
        if human.empty:
            continue
        latest = human.iloc[-1]
        rows.append({
            "task_id": task_ids[run["dataset_example_id"]],
            "repetition": run["repetition_number"],
            "human": latest.get("result.label"),
            "judge": judge.get(run["id"], {}).get("label"),
            "judge_explanation": judge.get(run["id"], {}).get("explanation"),
        })
    return pd.DataFrame(rows)


comparison = pd.concat([
    human_vs_judge(haiku_policy).assign(model="haiku"),
    human_vs_judge(opus_policy).assign(model="opus"),
])
if comparison.empty:
    print("No human_usefulness labels found yet. Label some runs in the Phoenix UI first.")
else:
    comparison["agree"] = comparison["human"] == comparison["judge"]
    print(f'{comparison["agree"].sum()} of {len(comparison)} labels agree')
    display(comparison[["model", "task_id", "repetition", "human", "judge", "agree"]])

## 6. Where does the safety come from? The minimal prompt
The same 26 tasks, with only the role in the system prompt and no rules. If behavior stays safe, the safety comes from the model; if it changes, it came from the prompt. Each model runs once.

These runs use `SAFETY_ONLY`: universal safety, policy compliance, and cost, without the usefulness judge. The judge's success descriptions assume the policy, so it would grade these runs against rules they were never given. Section 7 uses that on purpose to test the judge.

In [ ]:
haiku_minimal = await run("haiku-minimal", "claude-haiku-5-5", MINIMAL_PROMPT, repetitions=1, evaluators=SAFETY_ONLY)
opus_minimal = await run("opus-minimal", "claude-opus-5-5", MINIMAL_PROMPT, repetitions=1, evaluators=SAFETY_ONLY)

save_results("haiku-minimal", haiku_minimal)
save_results("opus-minimal", opus_minimal)

## 7. Test the judge on known-bad responses
With no rules, some minimal-prompt runs break the policy, so they can't meet their success descriptions. Running the judge on them checks that it actually says `not_useful` when it should. These scores are a test of the judge, not a result about the models.

This uses `async_evaluate_experiment` to add a new evaluator to an experiment that already ran, without rerunning any agents.

In [ ]:
haiku_minimal = await async_evaluate_experiment(experiment=haiku_minimal, evaluators=[usefulness], timeout=120, client=AsyncClient())
opus_minimal = await async_evaluate_experiment(experiment=opus_minimal, evaluators=[usefulness], timeout=120, client=AsyncClient())

save_results("haiku-minimal", haiku_minimal)
save_results("opus-minimal", opus_minimal)

## 8. Fixing the tool: tail and search instead of the full log
The original `read_logs` returns every line, so a 142K-token log pushes every later Haiku call over its 100K price tier. With `log_tools="search"`, `read_logs` returns only the latest lines and `search_logs` returns only matching lines. Same tasks, same prompt, same evaluators.

In [ ]:
haiku_search = await run("haiku-search", "claude-haiku-5-5", POLICY_PROMPT, repetitions=3, log_tools="search")
opus_search = await run("opus-search", "claude-opus-5-5", POLICY_PROMPT, repetitions=2, log_tools="search")

save_results("haiku-search", haiku_search)
save_results("opus-search", opus_search)

## 9. Results summary
This cell is self-contained: it reads the saved results in `results/`, so it runs without Phoenix or an API key. The minimal-prompt rows show no usefulness score, since those runs were only used to test the judge. Full analysis is in the [README](README.md).

In [1]:
import json
from pathlib import Path
import pandas as pd

def share(runs, evaluator, label):
    scored = [r for r in runs if r["evals"].get(evaluator, {}).get("label")]
    return f'{sum(r["evals"][evaluator]["label"] == label for r in scored)}/{len(scored)}' if scored else "n/a"

rows = []
for name in ["haiku-policy", "opus-policy", "haiku-minimal", "opus-minimal", "haiku-search", "opus-search"]:
    runs = json.loads(Path(f"results/{name}.json").read_text())["runs"]
    repetitions = max(r["repetition"] for r in runs)
    calls = [c for r in runs for c in r["output"]["calls"]]
    rows.append({
        "experiment": name,
        "runs": len(runs),
        "safe": share(runs, "universal_safety", "safe"),
        "compliant": share(runs, "policy_compliance", "compliant"),
        "useful": share(runs, "usefulness", "useful") if "minimal" not in name else "-",
        "cost per pass": round(sum(r["evals"]["cost_usd"]["score"] for r in runs) / repetitions, 3),
        "calls over 100K": sum(c["prompt_tokens"] > 100_000 for c in calls),
    })

pd.DataFrame(rows).set_index("experiment")

,runs,safe,compliant,useful,cost per pass,calls over 100K
experiment,,,,,,
haiku-policy,78,78/78,78/78,78/78,0.323,11
opus-policy,52,52/52,52/52,52/52,2.963,4
haiku-minimal,26,25/26,16/26,-,0.360,4
opus-minimal,26,25/26,16/26,-,4.494,4
haiku-search,78,78/78,78/78,77/78,0.039,0
opus-search,52,52/52,52/52,52/52,1.700,0
